### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="climate_model_weather_forecasting_1m",
    version_from_unique_name="climate_model_weather_forecasting",
    version_comment="""
We randomly sub-sample the train to 1 million and test data 250k rows. We follow TabReD and use random sub-sampling. The idea behind this instead of a time-based subsampling is to keep data from various time periods and model the distribution shift across the full time horizon.
""",
    # Same as climate_model_weather_forecasting.ipynb
    dataset_year="2024",
    domain_str="environmental science & climate",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/pcovkrd84mejm/tabred-weather",
    download_description="""
We get the TabRed data from Kaggle.

kaggle datasets download -d pcovkrd84mejm/tabred-weather -f weather.parquet && unzip weather.parquet.zip && rm weather.parquet.zip && mkdir -p local-data-warehouse/climate_model_weather_forecasting && mv weather.parquet local-data-warehouse/climate_model_weather_forecasting/
""",
    # References
    academic_reference_bibtex="""@inproceedings{rubachev2025tabred,
  title={TabReD: Analyzing Pitfalls and Filling the Gaps in Tabular Deep Learning Benchmarks},
  author={Rubachev, Ivan and Kartashev, Nikolay and Gorishniy, Yury and Babenko, Artem},
  booktitle={The Thirteenth International Conference on Learning Representations},
  year={2025},
}
""",
    academic_reference_bibtex_key="rubachev2025tabred",
    license="CC-BY-NC-SA-4.0",
    data_tags=["Non-IID", "Temporal"],
    curation_comments="""
We start with data from TabRed, which already comes preprocessed.

- This data is similar to data from https://arxiv.org/abs/2406.19380. This paper also describes the feature in more detail. The column `apply_time_rl` is new and might relate to a rolling lag/window indicator.
- We drop duplicated columns `cmc_0_1_66_0` , `cmc_0_1_67_0`, `cmc_0_1_68_0`.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="fact_temperature",
    problem_type="regression",
    objective_metric_name="rmse",
    time_on="fact_time",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_parquet(dataset_mold.path / "weather.parquet")
print("Loaded data shape:", df.shape)

Loaded data shape: (16951828, 104)


In [3]:
df["fact_time"] = pd.to_datetime(df['fact_time'] * 10**6, unit='us')

# We take all bin + cat as Category
cat_cols = [
    "fact_station_id",
    "cmc_available",
    "gfs_available",
    "gfs_soil_temperature_available",
]
df[cat_cols] = df[cat_cols].astype("category")

df = df.drop(columns=["cmc_0_1_66_0" , "cmc_0_1_67_0", "cmc_0_1_68_0"])

df = df.sort_values(by="fact_time").reset_index(drop=True)

## Data Checks

In [4]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 16,951,828
Columns: 101
Use sampling: True (sample size: 1,695,183)
Get missing and unique counts per column...


missing/unique per-col:   0%|          | 0/101 [00:00<?, ?it/s]

Get example values per column...


examples per-col:   0%|          | 0/101 [00:00<?, ?it/s]

Get numeric feature statistics...


...

/home/lennart_priorlabs_ai/.venvs/tabarena_1503/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['gfs_wind_speed', 'gfs_precipitable_water', 'cmc_0_2_2_500', 'cmc_0_2_2_700', 'gfs_u_wind', 'sun_elevation', 'cmc_0_2_2_850', 'cmc_0_2_2_925', 'gfs_v_wind', 'cmc_0_2_2_1000']
Rows remaining as candidates after top-10 filter: 2,222,967 (of 16,951,828)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...


hash cols:   0%|          | 0/101 [00:00<?, ?it/s]

group check:   0%|          | 0/101 [00:00<?, ?it/s]

Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [5]:
# Sample Rows
df_head

,apply_time_rl,climate_pressure,climate_temperature,cmc_0_0_0_1000,cmc_0_0_0_2,cmc_0_0_0_2_grad,cmc_0_0_0_2_interpolated,cmc_0_0_0_2_next,cmc_0_0_0_500,cmc_0_0_0_700,cmc_0_0_0_850,cmc_0_0_0_925,cmc_0_0_6_2,cmc_0_0_7_1000,cmc_0_0_7_2,cmc_0_0_7_500,cmc_0_0_7_700,cmc_0_0_7_850,cmc_0_0_7_925,cmc_0_1_0_0,cmc_0_1_11_0,cmc_0_1_65_0,cmc_0_1_7_0,cmc_0_2_2_10,cmc_0_2_2_1000,cmc_0_2_2_500,cmc_0_2_2_700,cmc_0_2_2_850,cmc_0_2_2_925,cmc_0_2_3_10,cmc_0_2_3_1000,cmc_0_2_3_500,cmc_0_2_3_700,cmc_0_2_3_850,cmc_0_2_3_925,cmc_0_3_0_0,cmc_0_3_0_0_next,cmc_0_3_1_0,cmc_0_3_5_1000,cmc_0_3_5_500,cmc_0_3_5_700,cmc_0_3_5_850,cmc_0_3_5_925,cmc_0_6_1_0,cmc_available,cmc_horizon_h,cmc_precipitations,cmc_timedelta_s,fact_latitude,fact_longitude,fact_station_id,fact_temperature,fact_time,gfs_2m_dewpoint,gfs_a_vorticity,gfs_available,gfs_cloudness,gfs_clouds_sea,gfs_horizon_h,gfs_humidity,gfs_precipitable_water,gfs_precipitations,gfs_pressure,gfs_r_velocity,gfs_soil_temperature,gfs_soil_temperature_available,gfs_temperature_10000,gfs_temperature_15000,gfs_temperature_20000,gfs_temperature_25000,gfs_temperature_30000,gfs_temperature_35000,gfs_temperature_40000,gfs_temperature_45000,gfs_temperature_5000,gfs_temperature_50000,gfs_temperature_55000,gfs_temperature_60000,gfs_temperature_65000,gfs_temperature_7000,gfs_temperature_70000,gfs_temperature_75000,gfs_temperature_80000,gfs_temperature_85000,gfs_temperature_90000,gfs_temperature_92500,gfs_temperature_95000,gfs_temperature_97500,gfs_temperature_sea,gfs_temperature_sea_grad,gfs_temperature_sea_interpolated,gfs_temperature_sea_next,gfs_timedelta_s,gfs_total_clouds_cover_high,gfs_total_clouds_cover_low,gfs_total_clouds_cover_middle,gfs_u_wind,gfs_v_wind,gfs_wind_speed,sun_elevation,topography_bathymetry
0,1656558028,734.744141,23.885000,298.649963,297.296814,1.407745,297.296814,298.704559,267.778595,283.189850,291.702332,293.898102,295.657227,2.625,1.375,2.125535,4.125,4.375,1.000,0.0182,0.0,8.422675,0.00000,0.877924,0.772647,-5.611496,5.720599,2.837402,0.612900,0.296210,0.220421,3.490018,4.065887,-1.089508,1.441672,97478.437500,97565.968750,100995.914062,87.548309,5869.221191,3148.391846,1500.609863,769.788940,56.0,1,36.0,0.011733,0.0,4.320160,118.127998,35268,25.0,2022-07-01,22.911097,0.000161,1,2.595,0.0,30.0,89.099998,58.275848,0.000000,751.914673,0.146159,25.669336,1,-82.938423,-67.529655,-51.575737,-39.137672,-29.290472,-21.140112,-14.710913,-9.310003,-65.094414,-4.954230,-1.558600,2.462945,6.070032,-72.122322,9.213800,12.611872,15.726191,18.676172,21.579950,22.924250,24.248041,25.053217,24.911097,2.334290,24.911097,27.245386,0.0,97.199997,62.700001,99.599998,1.331523,-1.190918,1.786404,27.538477,31.0
1,1656594029,682.137329,20.146429,297.659454,294.297913,2.450745,294.297913,296.748657,268.292236,282.191803,291.044312,294.509247,292.370850,3.875,1.750,2.625003,2.500,1.625,2.000,0.0156,0.0,0.241825,0.00000,-0.263098,-0.262909,-2.932794,6.427920,2.073233,-0.233868,0.026389,0.038190,2.432056,3.227490,0.622314,-0.060648,90688.929688,90709.109375,101027.851562,89.782349,5864.768555,3143.176025,1502.508789,771.345093,64.0,1,24.0,0.000000,0.0,3.733333,115.683334,46359,21.0,2022-07-01,17.850000,0.000008,1,1.820,0.0,18.0,94.699997,39.771606,0.166667,663.337524,0.003005,19.769342,1,-80.992363,-68.254646,-51.409691,-38.887978,-29.477819,-20.929022,-14.847906,-9.100012,-64.959373,-4.854132,-1.389471,2.203210,6.637231,-71.010078,9.605280,12.475275,15.475275,18.363306,19.663355,21.193354,22.683344,24.117121,18.697718,4.441895,18.697718,23.139612,0.0,100.000000,11.600000,70.400002,0.367437,0.011404,0.367613,25.068970,1509.0
2,1656432623,756.472900,23.574999,296.515198,296.540222,0.231506,296.540222,296.771729,267.050110,282.358337,285.808411,290.770325,292.365417,4.375,4.000,29.752676,16.625,4.375,1.625,0.0139,0.0,9.708450,0.00000,0.194861,-0.094766,2.726224,3.046457,-1.268372,-2.827625,5.378507,8.946704,6.409439,2.512552,14.478537,12.460943,101138.687500,101180.195312,101668.359375,145.335785,5864.513672,3154.175293,1

In [6]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,cmc_available,category,0.0,0.00,2.0,"1, 0"
1,fact_station_id,category,0.0,0.00,1948.0,"23179, 36164, 26758, 37249, 26525, 23220, 26519, 26778, 26505, 33172"
2,gfs_available,category,0.0,0.00,2.0,"1, 0"
3,gfs_soil_temperature_available,category,0.0,0.00,2.0,"1, 0"
4,fact_time,datetime64[ns],0.0,0.00,234962.0,"2023-06-13 12:00:00, 2023-06-15 12:00:00, 2023-05-21 12:00:00, 2023-07-04 12:00:00, 2023-06-19 12:00:00, 2023-05-16 12:00:00, 2023-06-14 12:00:00, 2023-05-31 12:00:00, 2023-05-25 12:00:00, 2023-06-04 12:00:00"
5,cmc_0_0_0_1000,float32,329334.0,1.94,721463.0,"299.858, 301.0854, 299.2025, 299.3289, 298.8618, 298.621, 299.5639, 300.5511, 299.6708, 299.4365"
6,cmc_0_0_0_2,float32,329334.0,1.94,770746.0,"298.8351, 301.1435, 301.7872, 301.5845, 301.758, 300.3935, 300.7822, 300.5709, 300.1096, 299.0441"
7,cmc_0_0_0_2_interpolated,float32,329334.0,1.94,800715.0,"301.7872, 299.6221, 300.4998, 300.1971, 298.5805, 298.3188, 301.8187, 300.5104, 298.7034, 298.8351"
8,cmc_0_0_0_2_next,float32,329334.0,1.94,760150.0,"300.35, 301.0845, 301.5324, 299.2493, 300.2383, 297.3351, 300.8372, 298.8935, 299.8945, 299.1583"
9,cmc_0_0_0_500,float32,329334.0,1.94,421011.0,"268.3295, 267.7184, 268.2761, 267.887, 268.1498, 268.2513, 268.0501, 268.1144, 267.9138, 268.387"


In [7]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
apply_time_rl,1695183.0,1.674947e+09,9.956256e+06,1.656381e+09,1.690743e+09
climate_pressure,1695183.0,7.437460e+02,2.576016e+01,5.547634e+02,7.752974e+02
climate_temperature,1695183.0,2.584963e+01,3.411622e+00,8.291429e+00,4.287500e+01
cmc_0_0_0_1000,1662082.0,2.996025e+02,3.461965e+00,2.728304e+02,3.171875e+02
cmc_0_0_0_2,1662082.0,2.992672e+02,3.866989e+00,2.720517e+02,3.168543e+02
cmc_0_0_0_2_grad,1695183.0,-1.951475e+02,1.383541e+03,-9.999000e+03,2.040555e+01
cmc_0_0_0_2_interpolated,1662082.0,2.993070e+02,3.773086e+00,2.717597e+02,3.168543e+02
cmc_0_0_0_2_next,1662082.0,2.993672e+02,3.801603e+00,2.710107e+02,3.169143e+02
cmc_0_0_0_500,1662082.0,2.678817e+02,1.804010e+00,2.479178e+02,2.762755e+02
cmc_0_0_0_700,1662082.0,2.830108e+02,1.711854e+00,2.648161e+02,2.914509e+02


In [8]:
# Categorical Feature Statistics
cat_stats

value    count    pct
column                         rank                                     
cmc_available                  1                       1  1662082  98.05
                               2                       0    33101   1.95
fact_station_id                1                   23179     3069   0.18
                               2                   36164     3049   0.18
                               3                   26758     3036   0.18
                               4                   37249     3035   0.18
                               5                   26525     3031   0.18
fact_time                      1     2023-06-13 12:00:00      505   0.03
                               2     2023-06-15 12:00:00      496   0.03
                               3     2023-05-21 12:00:00      494   0.03
                               4     2023-07-04 12:00:00      483   0.03
                               5     2023-06-19 12:00:00      481   0.03
gfs_available                  1                       1  1689951  99.69
                               2                       0     5232   0.31
gfs_soil_temperature_available 1                       1  1163952  68.66
                               2                       0   531231  31.34

In [9]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,-0.279,NaN,15.26,0.022,log1p,14551982.1,2.921207e+17,exponential


## Task Curation

In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import subsample_temporal

date_col = task_mold.time_on
target_col = task_mold.target_column_name

df = df.sort_values(by=date_col).reset_index(drop=True)

test_time_min = df[date_col].max().normalize() - pd.DateOffset(weeks=1)

# Define indices
test_idx = df.index[df[date_col] >= test_time_min].to_numpy().tolist()
train_idx = df.index[
    df[date_col] < test_time_min
].to_numpy().tolist()

df, train_idx, test_idx = subsample_temporal(
    df=df,
    train_idx=train_idx,
    test_idx=test_idx,
    stratify_on=task_mold.stratify_on,
)

print("Train size:", len(train_idx), "| Test size:", len(test_idx))
print("Train target mean:", df.loc[train_idx, target_col].mean())
print("Test target mean:", df.loc[test_idx, target_col].mean())

splits = {0: {0: (train_idx, test_idx)}}

splits_mold = PredictiveMLSplitsMetadata(
    # We diverge from TabRed here, as TabRed use 1 month. But we think 1 week is more realistic for weather forecasting as there rarely exist long term forecasts longer than 1 week in reality and thus the horizon might be too much.
    splits_comment="We use the last week as test data and all prior data as train data.",
    splits=splits,
    time_horizon=7,
    time_horizon_unit="days",
)

Train size: 1000000 | Test size: 250000
Train target mean: 26.876877
Test target mean: 27.206684


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to climate_model_weather_forecasting/versions/019d307b-f29b-7d7a-ada3-a8786a9bc50b
019d307b-f29b-7d7a-ada3-a8786a9bc50b
51accd3a387e9d01ed23f601f119e6ab2d49ee0826f14e63b4f10d0d0f725932
